# DVCM Physical Verification

This notebook reproduces the **independent physical checks** in `tests/test_dvcm_physical_verification.py` for the Discrete Vapor Cavity Model (DVCM). It is Binder-ready and mirrors the automated pytest tolerances so you can inspect charts and pass/fail metrics interactively.

**Checks**
1. **Mass conservation** — cavity volume step changes track bounded integration of `(Q_out - Q_in)` at the junction.
2. **Collapse spike** — the post-collapse head rise matches the discrete water-column collision estimate documented in `docs/dvcm_timestep_guidance.md`.

Launch in Binder: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Fdvcm_physical_verification.ipynb)

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

import rthym_moc

cwd = Path.cwd().resolve()
if (cwd / "tests").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "tests").exists():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

TESTS_DIR = REPO_ROOT / "tests"
if str(TESTS_DIR) not in sys.path:
    sys.path.insert(0, str(TESTS_DIR))

from dvcm_physical_verification_utils import (
    DEFAULT_DT_S,
    DEFAULT_P_VAPOR_PSI,
    evaluate_collapse_spike,
    evaluate_mass_conservation,
    junction_cavity_capacity_ft3,
    run_physical_verification_case,
    vapor_head_ft,
)

MASS_RTOL = 0.02
MASS_ATOL_FT3 = 1e-5
COLLAPSE_SPIKE_RTOL = 0.15

print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc version: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Run the column-separation transient

Symmetric reservoirs bound a central junction. Boundary heads drop below the vapor floor, then recover sharply to force cavity growth and collapse.

In [ ]:
results = run_physical_verification_case(dt=DEFAULT_DT_S)
time_s = np.asarray(results["time"], dtype=float)

head_j1 = np.asarray(results["node_head"]["J1"], dtype=float)
volume_j1 = np.asarray(results["node_cavity_volume"]["J1"], dtype=float)
active_j1 = np.asarray(results["node_cavity_active"]["J1"], dtype=int)
collapse_flag = np.asarray(results["node_cavity_collapse_flag"]["J1"], dtype=int)

q_in_gpm = np.asarray(results["pipe_flow_gpm"]["P1"], dtype=float)
q_out_gpm = np.asarray(results["pipe_flow_gpm"]["P2"], dtype=float)
q_net_cfs = (q_out_gpm - q_in_gpm) * rthym_moc.GPM_TO_CFS

print(f"Time steps: {len(time_s)}")
print(f"Peak junction head: {head_j1.max():.2f} ft")
print(f"Max cavity volume: {volume_j1.max():.6f} ft³")
print(f"Collapse events: {int(collapse_flag.sum())}")

## 3. Mass conservation invariant

In [ ]:
mass_metrics = evaluate_mass_conservation(
    results,
    dt=DEFAULT_DT_S,
    rtol=MASS_RTOL,
    atol_ft3=MASS_ATOL_FT3,
)

dt = DEFAULT_DT_S
max_step_delta = 0.25 * junction_cavity_capacity_ft3()
delta_v = np.diff(volume_j1)
expected_delta = np.sign(delta_v) * np.minimum(np.abs(q_net_cfs[1:]) * dt, max_step_delta)
mask = np.abs(delta_v) > 1e-12
step_error = np.full_like(delta_v, np.nan)
step_error[mask] = np.abs(delta_v[mask] - expected_delta[mask])

cumulative_flow = np.cumsum(q_net_cfs * dt)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(time_s, volume_j1, label="Tracked cavity volume", color="darkcyan", linewidth=2)
axes[0].plot(time_s, cumulative_flow, label="∫(Q_out − Q_in) dt (uncapped)", color="gray", linestyle="--")
axes[0].set_ylabel("Volume (ft³)")
axes[0].set_title("Cavity volume vs flow integral")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

axes[1].plot(time_s[1:][mask], step_error[mask], color="firebrick", linewidth=1.5)
axes[1].axhline(MASS_ATOL_FT3, color="black", linestyle=":", label=f"atol = {MASS_ATOL_FT3:g} ft³")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("|ΔV − expected| (ft³)")
axes[1].set_title("Per-step mass-balance error (DVCM cap applied in expected)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")
fig.tight_layout()
plt.show()

print("Mass conservation metrics")
print(f"  steps checked: {mass_metrics.n_steps_checked}")
print(f"  max abs error: {mass_metrics.max_abs_step_error_ft3:.3e} ft³")
print(f"  max rel error: {mass_metrics.max_rel_step_error:.3e}")
print(f"  PASS: {mass_metrics.passed} (rtol={MASS_RTOL}, atol={MASS_ATOL_FT3:g})")

## 4. Collapse spike verification

In [ ]:
spike_metrics = evaluate_collapse_spike(results, dt=DEFAULT_DT_S, rtol=COLLAPSE_SPIKE_RTOL)
h_vapor = vapor_head_ft(DEFAULT_P_VAPOR_PSI)
collapse_time = float(time_s[spike_metrics.collapse_step])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_s, head_j1, color="navy", linewidth=2, label="Junction HGL")
ax.axhline(h_vapor, color="red", linestyle="--", label=f"Vapor floor ({h_vapor:.1f} ft)")
ax.axhline(h_vapor + spike_metrics.expected_dh_ft, color="green", linestyle=":", label="Expected post-collapse rise")
ax.axvline(collapse_time, color="orange", linestyle="-.", label="Collapse step")
peak_idx = spike_metrics.collapse_step + int(
    np.argmax(head_j1[spike_metrics.collapse_step : spike_metrics.collapse_step + 3])
)
ax.scatter([time_s[peak_idx]], [head_j1[peak_idx]], color="gold", edgecolor="black", s=80, zorder=5, label="Observed peak")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Head (ft)")
ax.set_title("Secondary water-hammer spike after cavity collapse")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

print("Collapse spike metrics")
print(f"  collapse step: {spike_metrics.collapse_step} (t = {collapse_time:.3f} s)")
print(f"  V before collapse: {spike_metrics.v_before_ft3:.6f} ft³")
print(f"  observed ΔH: {spike_metrics.observed_dh_ft:.3f} ft")
print(f"  expected ΔH: {spike_metrics.expected_dh_ft:.3f} ft")
print(f"  relative error: {spike_metrics.relative_error:.3f}")
print(f"  PASS: {spike_metrics.passed} (rtol={COLLAPSE_SPIKE_RTOL})")

## 5. Summary

The table below mirrors the pytest assertions. The same tolerances are enforced in CI via `tests/test_dvcm_physical_verification.py`.

In [ ]:
summary = [
    ("Mass conservation (step deltas)", mass_metrics.passed, f"max rel err {mass_metrics.max_rel_step_error:.3e}"),
    ("Collapse spike (ΔH)", spike_metrics.passed, f"rel err {spike_metrics.relative_error:.3f}"),
    ("Cavity volume non-negative", bool(np.all(volume_j1 >= -1e-12)), ""),
    ("Cavity clears by end of run", bool(abs(volume_j1[-1]) < 1e-12), f"V_end = {volume_j1[-1]:.3e} ft³"),
]

print(f"{'Check':<36} {'PASS':<6} Details")
print("-" * 72)
for name, passed, detail in summary:
    print(f"{name:<36} {str(passed):<6} {detail}")

all_passed = all(row[1] for row in summary)
print("\nOverall:", "PASS" if all_passed else "FAIL")